# Standalone BEiT-3 Large 384 ingestion

All organizer keyframe archives stay temporarily on Colab SSD. Resumable vectors, metadata, and the final index are written to Drive. Use a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install faiss-cpu sentencepiece 'setuptools<81' timm torchscale transformers==4.46.3 torchmetrics==0.7.3 tensorboardX
!git clone --depth 1 https://github.com/microsoft/unilm.git /content/unilm
!sed -i 's/from torch._six import inf/from math import inf/' /content/unilm/beit3/utils.py
!wget -q https://github.com/addf400/files/releases/download/beit3/beit3_large_patch16_384_coco_retrieval.pth -O /content/beit3_large_patch16_384_coco_retrieval.pth
!wget -q https://github.com/addf400/files/releases/download/beit3/beit3.spm -O /content/beit3.spm
!mkdir -p /content/work
!wget -q https://aic-data.ledo.io.vn/map-keyframes-aic25-b1.zip -O /content/work/map-keyframes-aic25-b1.zip
!unzip -q /content/work/map-keyframes-aic25-b1.zip -d /content/work/mapping


In [ ]:
import csv
import shutil
import subprocess
import sys
from pathlib import Path

import faiss
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from transformers import XLMRobertaTokenizer

work = Path("/content/work")
output = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
features_dir = output / "beit3"
model_name = "beit3_large_patch16_384_coco_retrieval"
model_marker = features_dir / "model.txt"
if not model_marker.exists():
    shutil.rmtree(features_dir, ignore_errors=True)
    (output / "beit3.faiss").unlink(missing_ok=True)
    features_dir.mkdir(parents=True)
    model_marker.write_text(model_name, encoding="utf-8")
elif model_marker.read_text(encoding="utf-8").strip() != model_name:
    raise ValueError("clear AIC_ARTIFACTS/beit3 before changing BEiT-3 models")
metadata_path = output / "frames.csv"
mapping_dir = work / "mapping"
keyframe_archives = [
    ("Keyframes_L21.zip", "https://aic-data.ledo.io.vn/Keyframes_L21.zip"),
    ("Keyframes_L22.zip", "https://aic-data.ledo.io.vn/Keyframes_L22.zip"),
    ("Keyframes_L23.zip", "https://aic-data.ledo.io.vn/Keyframes_L23.zip"),
    ("Keyframes_L24.zip", "https://aic-data.ledo.io.vn/Keyframes_L24.zip"),
    ("Keyframes_L25.zip", "https://aic-data.ledo.io.vn/Keyframes_L25.zip"),
    ("Keyframes_L26_a.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_a.zip"),
    ("Keyframes_L26_b.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_b.zip"),
    ("Keyframes_L26_c.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_c.zip"),
    ("Keyframes_L26_d.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_d.zip"),
    ("Keyframes_L26_e.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip"),
    ("Keyframes_L27.zip", "https://aic-data.ledo.io.vn/Keyframes_L27.zip"),
    ("Keyframes_L28.zip", "https://aic-data.ledo.io.vn/Keyframes_L28.zip"),
    ("Keyframes_L29.zip", "https://aic-data.ledo.io.vn/Keyframes_L29.zip"),
    ("Keyframes_L30.zip", "https://aic-data.ledo.io.vn/Keyframes_L30.zip"),
]

device = "cuda" if torch.cuda.is_available() else "cpu"
sys.path.insert(0, "/content/unilm/beit3")
import modeling_finetune
import utils

model = modeling_finetune.beit3_large_patch16_384_retrieval()
utils.load_model_and_may_interpolate(
    "/content/beit3_large_patch16_384_coco_retrieval.pth", model, "model|module", ""
)
model = model.to(device).eval()
transform = transforms.Compose([
    transforms.Resize((384, 384), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
tokenizer = XLMRobertaTokenizer("/content/beit3.spm")


In [ ]:
def embed_images(paths, batch_size=4):
    vectors = []
    for start in range(0, len(paths), batch_size):
        images = []
        for path in paths[start : start + batch_size]:
            with Image.open(path) as image:
                images.append(transform(image.convert("RGB")))
        with torch.inference_mode():
            vision, _ = model(image=torch.stack(images).to(device), only_infer=True)
        vectors.append(vision.cpu().numpy().astype("float32"))
    return np.vstack(vectors)

metadata_videos = set()
next_frame_id = 0
if metadata_path.exists():
    with metadata_path.open(newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            if row["source"] == "organizer":
                metadata_videos.add(row["video_id"])
            next_frame_id = max(next_frame_id, int(row["frame_id"]) + 1)

for archive_name, url in keyframe_archives:
    archive_path = work / archive_name
    extract_dir = work / archive_name.removesuffix(".zip")
    subprocess.run(["wget", "-q", url, "-O", str(archive_path)], check=True)
    subprocess.run(["unzip", "-q", str(archive_path), "-d", str(extract_dir)], check=True)
    frame_dirs = []
    for path in extract_dir.rglob("L*_V*"):
        if path.is_dir() and any(path.glob("*.jpg")):
            frame_dirs.append(path)
    frame_dirs.sort(key=lambda path: path.name)
    for frame_dir in frame_dirs:
        video_id = frame_dir.name
        feature_path = features_dir / f"{video_id}.npy"
        if feature_path.exists():
            print(f"skip {video_id}")
        else:
            frames = sorted(frame_dir.glob("*.jpg"), key=lambda path: int(path.stem))
            vectors = embed_images(frames)
            np.save(feature_path, vectors)
            print(f"saved {video_id} {len(frames)} frames")
        if video_id in metadata_videos:
            continue
        frames = sorted(frame_dir.glob("*.jpg"), key=lambda path: int(path.stem))
        mapping_path = next(mapping_dir.rglob(f"{video_id}.csv"))
        timestamps = {}
        with mapping_path.open() as file:
            for row in csv.DictReader(file):
                timestamps[int(row["n"])] = float(row["pts_time"])
        write_header = not metadata_path.exists()
        with metadata_path.open("a", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=["frame_id", "frame_uid", "video_id", "keyframe_n", "timestamp_sec", "frame_path", "source"])
            if write_header:
                writer.writeheader()
            for frame in frames:
                keyframe_n = int(frame.stem)
                writer.writerow({
                    "frame_id": next_frame_id,
                    "frame_uid": f"{video_id}__kf_{keyframe_n}",
                    "video_id": video_id,
                    "keyframe_n": keyframe_n,
                    "timestamp_sec": timestamps[keyframe_n],
                    "frame_path": f"keyframes/keyframes/{video_id}/{frame.name}",
                    "source": "organizer",
                })
                next_frame_id += 1
        metadata_videos.add(video_id)
    shutil.rmtree(extract_dir)
    archive_path.unlink()


In [ ]:
rows_by_video = {}
with metadata_path.open(newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if row["source"] == "organizer":
            rows_by_video.setdefault(row["video_id"], []).append(row)

rows = []
index = None
for video_id, video_rows in rows_by_video.items():
    feature_path = features_dir / f"{video_id}.npy"
    if not feature_path.exists():
        continue
    vectors = np.load(feature_path, allow_pickle=False).astype("float32")
    if len(vectors) != len(video_rows):
        raise ValueError(f"{video_id}: feature/metadata mismatch")
    faiss.normalize_L2(vectors)
    if index is None:
        index = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1]))
    ids = np.array([int(row["frame_id"]) for row in video_rows], dtype="int64")
    index.add_with_ids(vectors, ids)
    rows.extend(video_rows)

faiss.write_index(index, str(output / "beit3.faiss"))
assert len(rows) == index.ntotal
print(f"indexed {index.ntotal} keyframes")


In [ ]:
query = "a person riding a bicycle"
tokens = tokenizer(query, max_length=64, padding="max_length", truncation=True, return_tensors="pt")["input_ids"].to(device)
with torch.inference_mode():
    _, text_vector = model(text_description=tokens, padding_mask=tokens.eq(tokenizer.pad_token_id), only_infer=True)
text_vector = text_vector.cpu().numpy().astype("float32")
faiss.normalize_L2(text_vector)
index = faiss.read_index(str(output / "beit3.faiss"))
with metadata_path.open(newline="", encoding="utf-8") as file:
    rows = {int(row["frame_id"]): row for row in csv.DictReader(file)}
scores, ids = index.search(text_vector, 5)
for score, row_id in zip(scores[0], ids[0]):
    row = rows[int(row_id)]
    print(row["video_id"], row["timestamp_sec"], float(score))
